# DSS Tomateiro — Treino YOLO-seg (Estágio A e Estágio B)

Notebook pronto para Google Colab seguindo estritamente o `SKILL.md` (seções 2 e 8):
* **Estágio A (`stage_a_leaf.pt`):** `0: leaf`, `1: ref_square` (`imgsz=640`, modelo `yolo11n-seg` ou `yolo11s-seg`)
* **Estágio B (`stage_b_lesion.pt`):** `0: necrosis`, `1: chlorosis` (`imgsz=1024`, modelo `yolo11s-seg`, recortes com fundo cinza 114)
* **GPU local alvo:** GTX 1660 Super (6 GB) — nunca usar modelos `m`, `l` ou `x`.
* **Validação:** divisão estritamente por grupo (planta/vaso/vídeo) e aceite por **CCC de Lin >= 0,90**.

In [ ]:
!pip -q install ultralytics opencv-python-headless numpy
import torch, ultralytics
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| Ultralytics:', ultralytics.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Google Drive e conferir estrutura dos datasets
Espera-se que `stage_a/` e `stage_b/` tenham sido preparados localmente com `vision_prep.py split` (garantindo zero vazamento de frames e fundo cinza 114 no estágio B).

In [ ]:
from pathlib import Path
# Descomente se estiver no Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/tomato_dss_datasets')
STAGE_A_YAML = DATA_ROOT / 'stage_a' / 'stage_a.yaml'
STAGE_B_YAML = DATA_ROOT / 'stage_b' / 'stage_b.yaml'
OUT_MODELS = DATA_ROOT / 'models'
OUT_MODELS.mkdir(parents=True, exist_ok=True)
print('Stage A YAML existe:', STAGE_A_YAML.is_file())
print('Stage B YAML existe:', STAGE_B_YAML.is_file())

## 2. Verificação anti-vazamento de grupos (treino vs validação)

In [ ]:
import re

POT_RE = re.compile(r'\b(V(?:0[1-9]|1\d|20))\b', re.IGNORECASE)

def group_of(name: str) -> str:
    stem = Path(name).stem
    m = POT_RE.search(stem.replace('-', ' ').replace('_', ' '))
    if m:
        return m.group(1).upper()
    return re.sub(r'_f\d+$', '', stem, flags=re.IGNORECASE)

for stage_dir in (DATA_ROOT / 'stage_a', DATA_ROOT / 'stage_b'):
    if not stage_dir.exists():
        continue
    tr = {group_of(p.name) for p in (stage_dir / 'images' / 'train').glob('*')}
    va = {group_of(p.name) for p in (stage_dir / 'images' / 'val').glob('*')}
    overlap = tr & va
    print(f'{stage_dir.name}: {len(tr)} grupos treino | {len(va)} grupos val | overlap = {overlap}')
    assert not overlap, f'ERRO: vazamento de grupos em {stage_dir.name}: {overlap}'

## 3. Treino do Estágio A (`leaf` + `ref_square`, `imgsz=640`)

In [ ]:
import shutil
from ultralytics import YOLO

model_a = YOLO('yolo11n-seg.pt')  # permitido: n ou s
res_a = model_a.train(
    data=str(STAGE_A_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    hsv_h=0.015,
    hsv_s=0.35,
    hsv_v=0.30,
    degrees=15.0,
    scale=0.35,
    fliplr=0.5,
    flipud=0.5,
    project='runs_tomato_dss',
    name='stage_a_leaf',
)
best_a = Path(res_a.save_dir) / 'weights' / 'best.pt'
shutil.copy2(best_a, OUT_MODELS / 'stage_a_leaf.pt')
print('Salvo:', OUT_MODELS / 'stage_a_leaf.pt')

## 4. Treino do Estágio B (`necrosis` + `chlorosis`, `imgsz=1024`, fundo cinza 114)

In [ ]:
model_b = YOLO('yolo11s-seg.pt')  # permitido: n ou s
res_b = model_b.train(
    data=str(STAGE_B_YAML),
    epochs=120,
    imgsz=1024,
    batch=8,
    patience=30,
    hsv_h=0.010,   # variação leve de matiz para não confundir clorose com verde sadio
    hsv_s=0.25,
    hsv_v=0.25,
    degrees=180.0,
    scale=0.30,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.3,
    project='runs_tomato_dss',
    name='stage_b_lesion',
)
best_b = Path(res_b.save_dir) / 'weights' / 'best.pt'
shutil.copy2(best_b, OUT_MODELS / 'stage_b_lesion.pt')
print('Salvo:', OUT_MODELS / 'stage_b_lesion.pt')

## 5. Validação por Coeficiente de Concordância de Lin (CCC >= 0,90)
Avalia o Estágio B nas imagens de validação comparando a severidade prevista com as máscaras manuais.

In [ ]:
import cv2
import numpy as np

def lin_ccc(y_true, y_pred):
    x, y = np.asarray(y_true, float), np.asarray(y_pred, float)
    mx, my = x.mean(), y.mean()
    vx, vy = x.var(), y.var()
    cov = np.mean((x - mx) * (y - my))
    return float(2 * cov / (vx + vy + (mx - my) ** 2))

trained_b = YOLO(str(OUT_MODELS / 'stage_b_lesion.pt'))
val_imgs = sorted((DATA_ROOT / 'stage_b' / 'images' / 'val').glob('*'))
y_true, y_pred = [], []

for img_p in val_imgs:
    img = cv2.imread(str(img_p))
    h, w = img.shape[:2]
    # Área foliar = pixels diferentes do fundo cinza 114
    leaf_px = int(np.any(img != 114, axis=2).sum())
    if leaf_px == 0:
        continue
    gt_mask = np.zeros((h, w), np.uint8)
    lbl_p = DATA_ROOT / 'stage_b' / 'labels' / 'val' / f'{img_p.stem}.txt'
    if lbl_p.is_file():
        for line in lbl_p.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 7:
                nums = [float(v) for v in parts[1:]]
                pts = np.array([[round(nums[i]*w), round(nums[i+1]*h)] for i in range(0, len(nums)-1, 2)], np.int32)
                cv2.fillPoly(gt_mask, [pts], 1)
    pred_res = trained_b.predict(img, imgsz=1024, conf=0.20, verbose=False)[0]
    pred_mask = np.zeros((h, w), np.uint8)
    if pred_res.masks is not None:
        for m in pred_res.masks.data.cpu().numpy():
            m_bin = cv2.resize((m > 0.5).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            pred_mask = np.maximum(pred_mask, m_bin)
    y_true.append(100.0 * gt_mask.sum() / leaf_px)
    y_pred.append(100.0 * pred_mask.sum() / leaf_px)

ccc = lin_ccc(y_true, y_pred)
print(f'N = {len(y_true)} | CCC de Lin = {ccc:.4f} | Meta >= 0.90: {ccc >= 0.90}')